# 01 — Data Audit, Cleaning & Exploratory Data Analysis

**Project:** Instamart Customer Intelligence & Support Copilot  
**Part A**

This notebook documents the actual Part A pipeline implemented in `src/audit.py`, `src/clean.py`, and `src/eda_part_a.py`. It inspects the raw CSV files, checks data-quality issues, runs the project's cleaning functions, stores cleaned tables, and explores the cleaned data.

**Learning goals**
- Load CSV data with Pandas and inspect rows, columns, types and missing values.
- Identify duplicates and invalid values before modelling.
- Apply the project's existing cleaning rules.
- Explore completed orders and basic business patterns.

> Run cells from top to bottom. The cleaning cell writes cleaned CSV files and refreshes `data/instamart.db`, just like the Python pipeline.

In [33]:
from pathlib import Path
import os, sys

# Find the project root whether this notebook is opened from the project folder or notebooks/.
CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Instamart project folder (or its notebooks folder).")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: d:\New folder\Instamart capstone project


## 1. Import libraries and project modules
Pandas is used for tabular data work. NumPy supports numeric missing-value handling. The project modules contain the canonical implementation.

In [34]:
import pandas as pd
import numpy as np
from src.audit import audit_raw_data
from src.clean import clean_customers, clean_orders, store_clean_data, run_cleaning_pipeline
from src.eda_part_a import run_eda

## 2. Load and inspect raw datasets
The raw folder contains customer, product, order, ticket, policy-evaluation and agent-scenario data. Start with the four operational CSVs used in Part A.

In [35]:
raw_dir = PROJECT_ROOT / "data" / "raw"
customers_raw = pd.read_csv(raw_dir / "customers.csv")
products_raw = pd.read_csv(raw_dir / "products.csv")
orders_raw = pd.read_csv(raw_dir / "orders.csv")
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for label, frame in [("Customers", customers_raw), ("Products", products_raw), ("Orders", orders_raw), ("Tickets", tickets_raw)]:
    print(f"{label}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
    display(frame.head())

Customers: 1,200 rows x 5 columns


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35.0,MALE
1,C0002,DELHI,2024-02-17,31.0,MALE
2,C0003,bengaluru,2025-02-18,18.0,F
3,C0004,Bengaluru,2024-06-21,24.0,FEMALE
4,C0005,Bengaluru,2024-10-01,25.0,FEMALE


Products: 40 rows x 4 columns


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245
3,P004,Sunflower Oil 1L,Grocery,155
4,P005,Sugar 1kg,Grocery,48


Orders: 13,084 rows x 9 columns


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-108250,C0792,P004,2,2025-09-01 12:24:59,completed,0,310.0,17.0
1,ORD-101939,C0190,P003,1,2025-08-17 07:07:23,completed,5,232.75,21.0
2,ORD-108765,C0841,P033,1,2025-07-13 09:42:32,completed,10,1349.1,27.0
3,ORD-108861,C0852,P010,1,2025-07-04 11:23:24,completed,0,45.0,30.0
4,ORD-109150,C0877,P018,2,2025-05-07 15:35:01,completed,20,96.0,28.0


Tickets: 164 rows x 5 columns


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical
3,T0159,C1172,,,none
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud


## 3. Data-quality inspection
Check data types, missing values, exact duplicate rows and numeric ranges. This helps identify problems before transformations.

In [36]:
def quality_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(dropna=True)
    })

display(quality_summary(customers_raw))
display(quality_summary(orders_raw))
print("Exact duplicate order rows:", orders_raw.duplicated().sum())
print("Non-numeric order amounts:", pd.to_numeric(orders_raw["order_amount"], errors="coerce").isna().sum())
print("Negative delivery minutes:", (pd.to_numeric(orders_raw["delivery_minutes"], errors="coerce") < 0).sum())

,dtype,missing_count,missing_percent,unique_values
customer_id,str,0,0.00,1200
city,str,0,0.00,24
signup_date,str,0,0.00,432
age,float64,61,5.08,49
gender,str,30,2.50,10


,dtype,missing_count,missing_percent,unique_values
order_id,str,0,0.00,12309
customer_id,str,0,0.00,1182
product_id,str,0,0.00,40
quantity,int64,0,0.00,3
order_ts,str,0,0.00,12959
status,str,0,0.00,3
discount_pct,int64,0,0.00,7
order_amount,str,53,0.41,499
delivery_minutes,float64,2541,19.42,69


Exact duplicate order rows: 119
Non-numeric order amounts: 120
Negative delivery minutes: 106


## 4. Run the project's raw-data audit
This function also compares observed counts with the reference checkpoints embedded in the project.

In [37]:
audit_results = audit_raw_data()
display(audit_results)

================ RAW DATA AUDIT REPORT ================

--- Table: customers.csv (Shape: 1200 rows, 5 cols) ---
               dtype  missing_count  missing_pct
customer_id      str              0         0.00
city             str              0         0.00
signup_date      str              0         0.00
age          float64             61         5.08
gender           str             30         2.50

--- Table: products.csv (Shape: 40 rows, 4 cols) ---
              dtype  missing_count  missing_pct
product_id      str              0          0.0
product_name    str              0          0.0
category        str              0          0.0
price_inr     int64              0          0.0

--- Table: orders.csv (Shape: 13084 rows, 9 cols) ---
                 dtype  missing_count  missing_pct
order_id           str              0         0.00
customer_id        str              0         0.00
product_id         str              0         0.00
quantity           str              0   

,Metric,Observed,Reference,Match
0,Raw orders.csv rows,13084,13084,True
1,Exact duplicate rows (orders.csv),119,119,True
2,Pending status rows (orders.csv),656,656,True
3,Distinct order_id values,12309,12309,True
4,Non-numeric order_amount count,120,120,True
5,Negative delivery_minutes count,106,106,True
6,Blank delivery_minutes count,2541,2541,True
7,Blank age count (customers.csv),61,61,True
8,Age outside 10-100 count (customers.csv),6,6,True
9,Distinct raw city spellings,24,24,True


## 5. Understand the cleaning rules
The existing project code applies these transformations:
- Customer city strings are trimmed and title-cased.
- Gender values are normalised to `male`, `female` or `unknown`.
- Age values outside 10–100 are treated as missing; missing ages are filled with the city median.
- Order timestamps are parsed; exact duplicate rows are removed; one latest row is retained per `order_id`.
- Invalid order amounts are filled using product price × quantity × (1 − discount%).
- Negative delivery times are treated as missing; completed orders use category medians for missing delivery times; cancelled orders remain missing.

In [38]:
# Run the canonical project cleaning pipeline. It saves data/clean/*.csv and data/instamart.db.
df_customers, df_products, df_orders = run_cleaning_pipeline()
print("Cleaned shapes:", df_customers.shape, df_products.shape, df_orders.shape)
display(df_customers.head())
display(df_orders.head())

Saved clean CSVs to data/clean/
Stored clean tables in SQLite database: data\instamart.db
Cleaned shapes: (1200, 5) (40, 4) (12309, 9)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female
3,C0004,Bengaluru,2024-06-21,24,female
4,C0005,Bengaluru,2024-10-01,25,female


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
1,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
2,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
3,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0
4,ORD-108115,C0776,P012,2,2025-04-01 08:02:58,completed,10,104.40,21.0


## 6. Validate cleaned data

In [39]:
print("Duplicate customer IDs:", df_customers["customer_id"].duplicated().sum())
print("Duplicate order IDs:", df_orders["order_id"].duplicated().sum())
print("Customer ages outside 10–100:", ((df_customers["age"] < 10) | (df_customers["age"] > 100)).sum())
print("Gender values:", sorted(df_customers["gender"].dropna().unique()))
print("Order status values:", sorted(df_orders["status"].dropna().unique()))

Duplicate customer IDs: 0
Duplicate order IDs: 0
Customer ages outside 10–100: 0
Gender values: ['female', 'male', 'unknown']
Order status values: ['cancelled', 'completed']


## 7. Exploratory data analysis
The next cell runs the project's EDA routine, which merges completed orders with customer and product details and reports summary statistics, outliers and correlations.

In [40]:
run_eda()

================ (A) PANDAS CROSS-CHECK ================

--- Q2 Pivot Table (Revenues in INR) ---
category       Dairy  Electronics    Grocery  Personal Care     Snacks       Total
city                                                                              
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   523643.40
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   463629.15
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   502829.80
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   528589.90
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   479058.50
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   500918.05
Total      531938.95   1219248.10  614293.30      168433.95  464754.50  2998668.80

Pivot Grand Total: INR 2,998,668.80 (Expected: INR 2,998,668.80) -> Match: True

--- City Completed Order Aggregations ---
           sum_order_amount  m

{'grand_total': np.float64(2998668.8),
 'city_agg':            sum_order_amount  mean_order_amount  count_order_amount
 city                                                              
 Bengaluru         523643.40             276.47                1894
 Chennai           463629.15             272.72                1700
 Delhi             502829.80             243.62                2064
 Hyderabad         528589.90             284.95                1855
 Mumbai            479058.50             297.00                1613
 Pune              500918.05             252.48                1984,
 'mean': np.float64(269.91),
 'median': np.float64(135.0),
 'skewness': np.float64(4.61),
 'q1': np.float64(64.8),
 'q3': np.float64(297.0),
 'upper_fence': np.float64(645.3),
 'outliers_above': np.int64(897),
 'outliers_below': np.int64(0),
 'corr_matrix':                   order_amount  discount_pct  quantity  delivery_minutes
 order_amount          1.000000     -0.095106  0.087294         -0.003135

## 8. Part A outputs
Expected project outputs include cleaned CSV files under `data/clean/` and a SQLite database at `data/instamart.db`. The EDA routine prints its analysis. Use the next cell to confirm the files exist.

In [41]:
for path in [PROJECT_ROOT/"data/clean/customers.csv", PROJECT_ROOT/"data/clean/products.csv", PROJECT_ROOT/"data/clean/orders.csv", PROJECT_ROOT/"data/instamart.db"]:
    print(path.relative_to(PROJECT_ROOT), "—", "FOUND" if path.exists() else "MISSING")

data\clean\customers.csv — FOUND
data\clean\products.csv — FOUND
data\clean\orders.csv — FOUND
data\instamart.db — FOUND


## 9. Interpretation
Write a short summary after running the notebook:
1. Which raw-data quality problems did the audit detect?
2. Which cleaning rule was most important for order data?
3. Why should missing delivery times for cancelled orders remain missing?
4. What business pattern did the EDA reveal?

**Part A complete:** raw data has been audited, cleaned, stored and explored.